# Proyecto analítico AquaLimpia

**Asignatura:** Ciencia de Datos  
**Unidad:** 3  
**Estudiante:** Fabiola Arrué Ravanal  

## Propósito del proyecto

Desarrollar un análisis reproducible de los datos operacionales y ambientales de AquaLimpia, evaluar su calidad y generar resultados diferenciados para las áreas de Operaciones y Gestión Ambiental.

## 1. Planteamiento analítico de la solución

AquaLimpia S. A. necesita describir y comparar el desempeño observado de sus plantas para identificar patrones, diferencias y situaciones que requieran revisión. Para responder a esta necesidad se propone un análisis descriptivo, exploratorio y comparativo, desarrollado mediante un flujo reproducible en Python.

El análisis considerará el caudal de entrada, la DBO de entrada y salida, la eficiencia de remoción, la energía de aireación, la generación de lodos y el cumplimiento registrado en el dataset. Los resultados se presentarán mediante visualizaciones interactivas y reportes diferenciados para Operaciones y Gestión Ambiental.

El alcance no incluye predicciones ni relaciones causales, porque el dataset no contiene antecedentes suficientes sobre capacidades de diseño, metas operacionales, condiciones del proceso o criterios normativos. Por esta razón, los resultados permitirán detectar comportamientos observados y orientar revisiones posteriores, pero no calificar por sí solos el funcionamiento de una planta ni verificar cumplimiento legal.

In [38]:
from copy import deepcopy
from pathlib import Path

import pandas as pd
import plotly.express as px

from plotly.subplots import make_subplots

import codigo.funciones_analisis as funciones

In [32]:
RUTA_DATOS = (
    Path("datos")
    / "dataset_set_A_aguas_residuales.xlsx"
)

if not RUTA_DATOS.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo: {RUTA_DATOS}"
    )

df_original = pd.read_excel(
    RUTA_DATOS
)

df = funciones.preparar_datos(
    df_original
)

print(
    f"Archivo cargado: {RUTA_DATOS}"
)

print(
    f"Filas originales: {df_original.shape[0]}"
)

print(
    f"Columnas originales: {df_original.shape[1]}"
)

print(
    f"Columnas preparadas: {df.shape[1]}"
)

df.head()

Archivo cargado: datos\dataset_set_A_aguas_residuales.xlsx
Filas originales: 200
Columnas originales: 10
Columnas preparadas: 12


,fecha_registro,planta,caudal_entrada_m3_d,DBO_entrada_mg_L,SST_entrada_mg_L,pH_entrada,energia_aeracion_kWh,lodos_generados_kg_d,DBO_salida_mg_L,cumplimiento_norma,eficiencia_remocion_DBO_pct,lodos_especificos_kg_m3
0,2025-08-17,Planta Sur,6562,271,324,7.12,1261.1,440.3,40.0,0,85.239852,0.067098
1,2025-09-07,Planta Sur,5336,322,230,7.01,1629.3,561.2,26.8,1,91.677019,0.105172
2,2025-07-26,Planta Norte,5755,318,282,6.79,1469.2,441.6,45.2,0,85.786164,0.076733
3,2025-10-27,Planta Centro,6840,216,167,7.01,1764.8,560.0,38.0,0,82.407407,0.081871
4,2025-09-06,Planta Centro,6803,326,200,7.55,1409.3,654.0,32.1,0,90.153374,0.096134


## 2. Inspección inicial del dataset

In [3]:
resumen_estructura = pd.DataFrame({
    "tipo_dato": df.dtypes.astype(str),
    "valores_no_nulos": df.notna().sum(),
    "valores_nulos": df.isna().sum(),
    "valores_unicos": df.nunique()
})

resumen_estructura

,tipo_dato,valores_no_nulos,valores_nulos,valores_unicos
fecha_registro,str,200,0,98
planta,str,200,0,3
caudal_entrada_m3_d,int64,200,0,191
DBO_entrada_mg_L,int64,200,0,143
SST_entrada_mg_L,int64,200,0,138
pH_entrada,float64,200,0,109
energia_aeracion_kWh,float64,200,0,198
lodos_generados_kg_d,float64,200,0,199
DBO_salida_mg_L,float64,200,0,167
cumplimiento_norma,int64,200,0,2


In [33]:
print(
    "Tipo de dato:",
    df["fecha_registro"].dtype
)

print(
    "Fecha inicial:",
    df["fecha_registro"].min().date()
)

print(
    "Fecha final:",
    df["fecha_registro"].max().date()
)

Tipo de dato: datetime64[us]
Fecha inicial: 2025-07-01
Fecha final: 2025-10-28


In [5]:
resumen_calidad = pd.Series({
    "cantidad_filas": len(df),
    "cantidad_columnas": df.shape[1],
    "valores_nulos_totales": int(df.isna().sum().sum()),
    "filas_duplicadas": int(df.duplicated().sum()),
    "fechas_unicas": df["fecha_registro"].nunique(),
    "plantas_unicas": df["planta"].nunique()
}, name="resultado").to_frame()

resumen_calidad

,resultado
cantidad_filas,200
cantidad_columnas,10
valores_nulos_totales,0
filas_duplicadas,0
fechas_unicas,98
plantas_unicas,3


In [6]:
registros_por_fecha_planta = (
    df.groupby(["fecha_registro", "planta"])
      .size()
      .reset_index(name="cantidad_registros")
)

combinaciones_repetidas = registros_por_fecha_planta[
    registros_por_fecha_planta["cantidad_registros"] > 1
]

print(
    "Combinaciones fecha-planta con más de un registro:",
    len(combinaciones_repetidas)
)

print(
    "Máximo de registros para una misma fecha y planta:",
    registros_por_fecha_planta["cantidad_registros"].max()
)

combinaciones_repetidas.head(10)

Combinaciones fecha-planta con más de un registro: 38
Máximo de registros para una misma fecha y planta: 3


,fecha_registro,planta,cantidad_registros
4,2025-07-05,Planta Centro,2
8,2025-07-07,Planta Norte,2
11,2025-07-08,Planta Sur,2
12,2025-07-09,Planta Sur,2
16,2025-07-13,Planta Centro,2
26,2025-07-20,Planta Norte,2
30,2025-07-22,Planta Sur,2
31,2025-07-24,Planta Centro,2
32,2025-07-24,Planta Norte,2
37,2025-07-30,Planta Centro,3


In [7]:
registros_adicionales = int(
    (combinaciones_repetidas["cantidad_registros"] - 1).sum()
)

print(
    "Registros adicionales dentro de combinaciones repetidas:",
    registros_adicionales
)

Registros adicionales dentro de combinaciones repetidas: 44


### 2.1 Resultados de la inspección inicial

El dataset contiene 200 registros y 10 variables correspondientes a tres plantas. No se identificaron valores nulos ni filas completamente duplicadas. La información abarca desde el 1 de julio hasta el 28 de octubre de 2025, periodo de 120 días que comprende partes de cuatro meses calendario, aunque el caso lo presenta como el último trimestre.

Se identificaron 98 fechas únicas y 38 combinaciones fecha-planta con más de una observación. Estas combinaciones reúnen 44 registros adicionales posteriores al primero y alcanzan un máximo de tres observaciones para una misma planta y fecha. Los registros no son duplicados completos, pero el dataset no incluye hora ni identificador de muestra. Esta limitación impide establecer el orden intradía y exige resumir con cautela la información temporal, sin eliminar observaciones cuya diferencia podría ser válida.

In [8]:
validaciones_basicas = pd.Series({
    "caudal_no_positivo": (
        df["caudal_entrada_m3_d"] <= 0
    ).sum(),
    "DBO_entrada_no_positiva": (
        df["DBO_entrada_mg_L"] <= 0
    ).sum(),
    "SST_entrada_no_positiva": (
        df["SST_entrada_mg_L"] <= 0
    ).sum(),
    "energia_negativa": (
        df["energia_aeracion_kWh"] < 0
    ).sum(),
    "lodos_negativos": (
        df["lodos_generados_kg_d"] < 0
    ).sum(),
    "pH_fuera_de_rango_0_14": (
        (df["pH_entrada"] < 0) |
        (df["pH_entrada"] > 14)
    ).sum(),
    "DBO_salida_mayor_que_entrada": (
        df["DBO_salida_mg_L"] > df["DBO_entrada_mg_L"]
    ).sum(),
    "cumplimiento_fuera_de_0_1": (
        ~df["cumplimiento_norma"].isin([0, 1])
    ).sum()
}, name="cantidad").to_frame()

validaciones_basicas

,cantidad
caudal_no_positivo,0
DBO_entrada_no_positiva,0
SST_entrada_no_positiva,0
energia_negativa,0
lodos_negativos,0
pH_fuera_de_rango_0_14,0
DBO_salida_mayor_que_entrada,0
cumplimiento_fuera_de_0_1,0


## 3. Construcción de variables derivadas

Para comparar el desempeño observado se calculará la eficiencia de remoción de DBO como la diferencia entre la concentración de entrada y salida, dividida por la concentración de entrada. También se calculará la generación específica de lodos mediante la relación entre los kilogramos diarios de lodos y el caudal tratado diariamente.

No se calculará energía específica porque el nombre de la variable `energia_aeracion_kWh` no identifica el periodo asociado al consumo. Sin esta información no es posible asegurar que su división por el caudal diario produzca una medida válida en kWh/m³.

Las variables derivadas se utilizarán con fines descriptivos y comparativos. Sus resultados no establecerán por sí solos que una planta funciona correctamente o presenta una deficiencia, debido a la ausencia de metas operacionales, capacidades de diseño y rangos técnicos de referencia.

In [34]:
columnas_derivadas = [
    "fecha_registro",
    "planta",
    "DBO_entrada_mg_L",
    "DBO_salida_mg_L",
    "eficiencia_remocion_DBO_pct",
    "lodos_especificos_kg_m3"
]

columnas_faltantes = [
    columna
    for columna in columnas_derivadas
    if columna not in df.columns
]

if columnas_faltantes:
    raise ValueError(
        "Faltan variables preparadas: "
        + ", ".join(columnas_faltantes)
    )

vista_variables_derivadas = (
    df[columnas_derivadas]
    .head()
    .copy()
)

columnas_numericas_derivadas = [
    "DBO_entrada_mg_L",
    "DBO_salida_mg_L",
    "eficiencia_remocion_DBO_pct",
    "lodos_especificos_kg_m3"
]

vista_variables_derivadas[
    columnas_numericas_derivadas
] = vista_variables_derivadas[
    columnas_numericas_derivadas
].round(2)

vista_variables_derivadas

,fecha_registro,planta,DBO_entrada_mg_L,DBO_salida_mg_L,eficiencia_remocion_DBO_pct,lodos_especificos_kg_m3
0,2025-08-17,Planta Sur,271,40.0,85.24,0.07
1,2025-09-07,Planta Sur,322,26.8,91.68,0.11
2,2025-07-26,Planta Norte,318,45.2,85.79,0.08
3,2025-10-27,Planta Centro,216,38.0,82.41,0.08
4,2025-09-06,Planta Centro,326,32.1,90.15,0.10


## 4. Análisis descriptivo y comparativo por planta

La comparación se realizará considerando la cantidad de observaciones de cada planta, medidas de tendencia central y variables operacionales y ambientales. Se utilizarán el promedio y la mediana para describir el comportamiento central, junto con medidas de dispersión y visualizaciones que permitan identificar variabilidad y posibles valores atípicos.

In [10]:
resumen_por_planta = (
    df.groupby("planta", as_index=False)
      .agg(
          registros=(
              "planta",
              "size"
          ),
          caudal_promedio_m3_d=(
              "caudal_entrada_m3_d",
              "mean"
          ),
          DBO_salida_promedio_mg_L=(
              "DBO_salida_mg_L",
              "mean"
          ),
          DBO_salida_mediana_mg_L=(
              "DBO_salida_mg_L",
              "median"
          ),
          eficiencia_promedio_pct=(
              "eficiencia_remocion_DBO_pct",
              "mean"
          ),
          eficiencia_mediana_pct=(
              "eficiencia_remocion_DBO_pct",
              "median"
          ),
          cumplimiento_registrado_pct=(
              "cumplimiento_norma",
              lambda serie: serie.mean() * 100
          ),
          lodos_especificos_mediana_kg_m3=(
              "lodos_especificos_kg_m3",
              "median"
          )
      )
)

resumen_por_planta.round(2)

,planta,registros,caudal_promedio_m3_d,DBO_salida_promedio_mg_L,DBO_salida_mediana_mg_L,eficiencia_promedio_pct,eficiencia_mediana_pct,cumplimiento_registrado_pct,lodos_especificos_mediana_kg_m3
0,Planta Centro,75,5112.72,35.90,35.50,87.51,87.76,22.67,0.09
1,Planta Norte,71,5287.87,36.56,34.20,86.65,86.72,16.90,0.09
2,Planta Sur,54,4684.52,36.06,34.65,87.10,86.73,29.63,0.08


### 4.1 Comparación inicial

La cantidad de registros no es igual entre plantas: Centro contiene 75 observaciones, Norte 71 y Sur 54. Por esta razón, las comparaciones se basan en promedios, medianas y proporciones, mientras que los tamaños muestrales se conservan para interpretar el respaldo de cada estimación.

Las eficiencias promedio de remoción son cercanas. Planta Centro presenta el mayor valor observado, con 87,51 %, y Planta Norte el menor, con 86,65 %. Esta diferencia es inferior a un punto porcentual y no permite afirmar por sí sola que una instalación funciona mejor que otra.

Las tres plantas registran una DBO de salida promedio próxima a 36 mg/L. Sin embargo, las proporciones de cumplimiento registrado varían desde 16,90 % en Planta Norte hasta 29,63 % en Planta Sur. Estos resúmenes no permiten reconstruir la regla utilizada para clasificar el cumplimiento, por lo que esta variable se analizará como una etiqueta incluida en el dataset y no como una verificación normativa.

In [11]:
comparacion_eficiencia = resumen_por_planta.melt(
    id_vars="planta",
    value_vars=[
        "eficiencia_promedio_pct",
        "eficiencia_mediana_pct"
    ],
    var_name="medida",
    value_name="eficiencia_pct"
)

comparacion_eficiencia["medida"] = (
    comparacion_eficiencia["medida"]
    .replace({
        "eficiencia_promedio_pct": "Promedio",
        "eficiencia_mediana_pct": "Mediana"
    })
)

fig_eficiencia = px.bar(
    comparacion_eficiencia,
    x="planta",
    y="eficiencia_pct",
    color="medida",
    barmode="group",
    text_auto=".2f",
    title="Eficiencia de remoción de DBO por planta",
    labels={
        "planta": "Planta",
        "eficiencia_pct": "Eficiencia de remoción (%)",
        "medida": "Medida"
    },
    color_discrete_map={
        "Promedio": "#4C9BE8",
        "Mediana": "#F2A65A"
    },
    template="plotly_dark"
)

fig_eficiencia.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig_eficiencia.update_yaxes(
    range=[0, 100]
)

fig_eficiencia.update_layout(
    height=500,
    legend_title_text=""
)

fig_eficiencia.show()

In [12]:
fig_distribucion_eficiencia = px.box(
    df,
    x="planta",
    y="eficiencia_remocion_DBO_pct",
    color="planta",
    points="outliers",
    title="Distribución de la eficiencia de remoción de DBO",
    labels={
        "planta": "Planta",
        "eficiencia_remocion_DBO_pct": "Eficiencia de remoción (%)"
    },
    hover_data={
        "fecha_registro": True,
        "DBO_entrada_mg_L": ":.1f",
        "DBO_salida_mg_L": ":.1f"
    },
    template="plotly_dark"
)

fig_distribucion_eficiencia.update_layout(
    height=500,
    showlegend=False
)

fig_distribucion_eficiencia.show()

In [13]:
eficiencia_diaria = (
    df.groupby(
        ["fecha_registro", "planta"],
        as_index=False
    )
    .agg(
        eficiencia_mediana_pct=(
            "eficiencia_remocion_DBO_pct",
            "median"
        ),
        cantidad_registros=(
            "eficiencia_remocion_DBO_pct",
            "size"
        )
    )
)

fig_eficiencia_temporal = px.line(
    eficiencia_diaria,
    x="fecha_registro",
    y="eficiencia_mediana_pct",
    color="planta",
    markers=True,
    title="Evolución de la eficiencia mediana de remoción por fecha",
    labels={
        "fecha_registro": "Fecha",
        "eficiencia_mediana_pct": "Eficiencia mediana (%)",
        "planta": "Planta"
    },
    hover_data={
        "cantidad_registros": True
    },
    template="plotly_dark"
)

fig_eficiencia_temporal.update_layout(
    height=550,
    legend_title_text=""
)

fig_eficiencia_temporal.update_xaxes(
    tickformat="%d-%m-%Y"
)

fig_eficiencia_temporal.show()

In [14]:
clasificaciones_por_DBO = (
    df.groupby("DBO_salida_mg_L")["cumplimiento_norma"]
      .nunique()
)

valores_con_clasificacion_mixta = (
    clasificaciones_por_DBO[
        clasificaciones_por_DBO > 1
    ]
    .index
    .tolist()
)

registros_clasificacion_mixta = (
    df[
        df["DBO_salida_mg_L"].isin(
            valores_con_clasificacion_mixta
        )
    ]
    [
        [
            "fecha_registro",
            "planta",
            "DBO_salida_mg_L",
            "cumplimiento_norma"
        ]
    ]
    .sort_values(
        ["DBO_salida_mg_L", "cumplimiento_norma"]
    )
)

print(
    "Valores de DBO de salida presentes en ambas clasificaciones:",
    valores_con_clasificacion_mixta
)

registros_clasificacion_mixta

Valores de DBO de salida presentes en ambas clasificaciones: [21.3, 24.7, 25.4, 25.6, 26.4]


,fecha_registro,planta,DBO_salida_mg_L,cumplimiento_norma
11,2025-07-24,Planta Norte,21.3,0
36,2025-08-23,Planta Norte,21.3,1
81,2025-07-20,Planta Norte,24.7,0
65,2025-10-14,Planta Norte,24.7,1
143,2025-09-30,Planta Centro,25.4,0
66,2025-08-05,Planta Sur,25.4,1
129,2025-08-09,Planta Centro,25.6,0
142,2025-08-14,Planta Centro,25.6,1
96,2025-07-19,Planta Centro,26.4,0
15,2025-08-12,Planta Centro,26.4,1


### 4.2 Cumplimiento registrado y DBO de salida

La variable `cumplimiento_norma` se analizará como una clasificación proporcionada por el dataset. No se interpretará como una verificación legal, porque no se dispone de la norma aplicada, los criterios complementarios ni la metodología utilizada para asignar cada etiqueta.

Se identificaron cinco valores de DBO de salida presentes tanto en registros clasificados con cumplimiento como en registros clasificados con incumplimiento: 21,3; 24,7; 25,4; 25,6 y 26,4 mg/L. Este hallazgo no demuestra que las etiquetas sean erróneas, pero confirma que la concentración de salida no permite explicar por sí sola la clasificación registrada. Por esta razón, los resultados se presentarán sin sustituir, corregir ni reinterpretar las etiquetas originales.

In [15]:
fig_cumplimiento = px.bar(
    resumen_por_planta,
    x="planta",
    y="cumplimiento_registrado_pct",
    color="planta",
    text_auto=".2f",
    title="Proporción de cumplimiento registrado por planta",
    labels={
        "planta": "Planta",
        "cumplimiento_registrado_pct":
            "Cumplimiento registrado (%)"
    },
    template="plotly_dark"
)

fig_cumplimiento.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig_cumplimiento.update_yaxes(
    range=[0, 100]
)

fig_cumplimiento.update_layout(
    height=500,
    showlegend=False
)

fig_cumplimiento.show()

In [16]:
fig_caudal_energia = px.scatter(
    df,
    x="caudal_entrada_m3_d",
    y="energia_aeracion_kWh",
    color="planta",
    opacity=0.75,
    title="Relación entre caudal de entrada y energía de aireación",
    labels={
        "caudal_entrada_m3_d": "Caudal de entrada (m³/día)",
        "energia_aeracion_kWh": "Energía de aireación registrada (kWh)",
        "planta": "Planta"
    },
    hover_data={
        "fecha_registro": True,
        "caudal_entrada_m3_d": ":.0f",
        "energia_aeracion_kWh": ":.1f"
    },
    template="plotly_dark"
)

fig_caudal_energia.update_layout(
    height=550,
    legend_title_text=""
)

fig_caudal_energia.show()

In [17]:
resultados_correlacion = []

for planta, grupo in df.groupby("planta"):
    resultados_correlacion.append({
        "planta": planta,
        "correlacion_Pearson": grupo[
            "caudal_entrada_m3_d"
        ].corr(
            grupo["energia_aeracion_kWh"],
            method="pearson"
        ),
        "correlacion_Spearman": grupo[
            "caudal_entrada_m3_d"
        ].corr(
            grupo["energia_aeracion_kWh"],
            method="spearman"
        )
    })

correlacion_caudal_energia = pd.DataFrame(
    resultados_correlacion
)

correlacion_caudal_energia.round(3)

,planta,correlacion_Pearson,correlacion_Spearman
0,Planta Centro,0.832,0.869
1,Planta Norte,0.863,0.835
2,Planta Sur,0.873,0.838


### 4.3 Relación entre caudal y energía de aireación

En esta muestra se observa una asociación positiva fuerte entre el caudal de entrada y la energía de aireación registrada en las tres plantas. Los coeficientes de Pearson se encuentran entre 0,832 y 0,873, mientras que los coeficientes de Spearman varían entre 0,835 y 0,869.

Estos resultados indican que los registros con mayor caudal tienden a presentar valores superiores de energía de aireación. Sin embargo, la asociación no demuestra que el aumento del caudal sea la única causa del consumo energético ni permite comparar eficiencia entre plantas. Además, el dataset no especifica el periodo asociado a la variable de energía, por lo que no se calculará un indicador en kWh/m³ sin validación adicional.

In [18]:
fig_lodos_especificos = px.box(
    df,
    x="planta",
    y="lodos_especificos_kg_m3",
    color="planta",
    points="outliers",
    title="Distribución de la generación específica de lodos",
    labels={
        "planta": "Planta",
        "lodos_especificos_kg_m3":
            "Generación específica de lodos (kg/m³)"
    },
    hover_data={
        "fecha_registro": True,
        "caudal_entrada_m3_d": ":.0f",
        "lodos_generados_kg_d": ":.1f"
    },
    template="plotly_dark"
)

fig_lodos_especificos.update_layout(
    height=500,
    showlegend=False
)

fig_lodos_especificos.show()

In [19]:
fig_DBO_salida = px.box(
    df,
    x="planta",
    y="DBO_salida_mg_L",
    color="planta",
    points="outliers",
    title="Distribución de la DBO de salida por planta",
    labels={
        "planta": "Planta",
        "DBO_salida_mg_L": "DBO de salida (mg/L)"
    },
    hover_data={
        "fecha_registro": True,
        "DBO_entrada_mg_L": ":.1f",
        "cumplimiento_norma": True
    },
    template="plotly_dark"
)

fig_DBO_salida.update_layout(
    height=500,
    showlegend=False
)

fig_DBO_salida.show()

In [21]:
candidatos_DBO_salida = []

for planta, grupo in df.groupby("planta"):
    q1 = grupo["DBO_salida_mg_L"].quantile(0.25)
    q3 = grupo["DBO_salida_mg_L"].quantile(0.75)
    rango_intercuartil = q3 - q1

    limite_inferior = q1 - 1.5 * rango_intercuartil
    limite_superior = q3 + 1.5 * rango_intercuartil

    candidatos = grupo[
        (grupo["DBO_salida_mg_L"] < limite_inferior) |
        (grupo["DBO_salida_mg_L"] > limite_superior)
    ].copy()

    candidatos["limite_inferior"] = limite_inferior
    candidatos["limite_superior"] = limite_superior

    candidatos_DBO_salida.append(candidatos)

candidatos_DBO_salida = pd.concat(
    candidatos_DBO_salida,
    ignore_index=True
)

columnas_revision_DBO = [
    "fecha_registro",
    "planta",
    "DBO_entrada_mg_L",
    "DBO_salida_mg_L",
    "cumplimiento_norma",
    "limite_inferior",
    "limite_superior"
]

print(
    "Cantidad de candidatos atípicos:",
    len(candidatos_DBO_salida)
)

tabla_candidatos_DBO = candidatos_DBO_salida[
    columnas_revision_DBO
].copy()

columnas_numericas_revision = [
    "DBO_entrada_mg_L",
    "DBO_salida_mg_L",
    "limite_inferior",
    "limite_superior"
]

tabla_candidatos_DBO[
    columnas_numericas_revision
] = tabla_candidatos_DBO[
    columnas_numericas_revision
].round(2)

tabla_candidatos_DBO

Cantidad de candidatos atípicos: 2


,fecha_registro,planta,DBO_entrada_mg_L,DBO_salida_mg_L,cumplimiento_norma,limite_inferior,limite_superior
0,2025-07-01,Planta Norte,432,79.0,0,-3.45,74.15
1,2025-10-25,Planta Norte,432,76.8,0,-3.45,74.15


### 4.4 Revisión de posibles valores atípicos

El criterio del rango intercuartílico identificó dos candidatos atípicos en la DBO de salida de Planta Norte. Los registros corresponden al 1 de julio y al 25 de octubre de 2025, con concentraciones de 79,0 y 76,8 mg/L, respectivamente. Ambos valores superan el límite estadístico superior de 74,15 mg/L calculado para esa planta.

Estas observaciones no presentan valores nulos, no corresponden a filas duplicadas y mantienen una DBO de salida inferior a la concentración de entrada. Por esta razón, se conservarán en el análisis. El criterio estadístico permite identificar registros que requieren revisión, pero no demuestra que exista un error de medición ni justifica su eliminación automática.

## 5. Dashboard exploratorio

El dashboard integra los principales resultados operacionales y ambientales para facilitar la comparación entre plantas. Incluye la eficiencia de remoción de DBO, la concentración de salida, el cumplimiento registrado, la relación entre caudal y energía de aireación y la evolución temporal de los indicadores.

Las visualizaciones conservan el detalle disponible mediante información emergente y herramientas de exploración. Los resultados describen el comportamiento de la muestra y deben interpretarse junto con las limitaciones de calidad, granularidad y contexto técnico identificadas durante el análisis.

In [39]:
dashboard = make_subplots(
    rows=3,
    cols=2,
    specs=[
        [{}, {}],
        [{}, {}],
        [{"colspan": 2}, None]
    ],
    subplot_titles=[
        "Eficiencia promedio y mediana",
        "Cumplimiento registrado",
        "Caudal y energía de aireación",
        "Distribución de la DBO de salida",
        "Evolución temporal de la eficiencia"
    ],
    vertical_spacing=0.10,
    horizontal_spacing=0.10
)


def agregar_trazas(
    figura_origen,
    fila,
    columna,
    mostrar_leyenda
):
    for traza in figura_origen.data:
        nueva_traza = deepcopy(traza)
        nueva_traza.showlegend = mostrar_leyenda

        dashboard.add_trace(
            nueva_traza,
            row=fila,
            col=columna
        )


agregar_trazas(
    fig_eficiencia,
    fila=1,
    columna=1,
    mostrar_leyenda=True
)

agregar_trazas(
    fig_cumplimiento,
    fila=1,
    columna=2,
    mostrar_leyenda=False
)

agregar_trazas(
    fig_caudal_energia,
    fila=2,
    columna=1,
    mostrar_leyenda=True
)

agregar_trazas(
    fig_DBO_salida,
    fila=2,
    columna=2,
    mostrar_leyenda=False
)

agregar_trazas(
    fig_eficiencia_temporal,
    fila=3,
    columna=1,
    mostrar_leyenda=False
)

dashboard.update_yaxes(
    title_text="Eficiencia (%)",
    range=[0, 100],
    row=1,
    col=1
)

dashboard.update_yaxes(
    title_text="Cumplimiento registrado (%)",
    range=[0, 100],
    row=1,
    col=2
)

dashboard.update_xaxes(
    title_text="Caudal de entrada (m³/día)",
    row=2,
    col=1
)

dashboard.update_yaxes(
    title_text="Energía registrada (kWh)",
    row=2,
    col=1
)

dashboard.update_yaxes(
    title_text="DBO de salida (mg/L)",
    row=2,
    col=2
)

dashboard.update_xaxes(
    title_text="Fecha",
    tickformat="%d-%m-%Y",
    rangeslider_visible=True,
    row=3,
    col=1
)

dashboard.update_yaxes(
    title_text="Eficiencia mediana (%)",
    row=3,
    col=1
)

dashboard.update_layout(
    title={
        "text": "Dashboard exploratorio de AquaLimpia S. A.",
        "x": 0.5,
        "xanchor": "center"
    },
    template="plotly_dark",
    height=1250,
    barmode="group",
    boxmode="group",
    hovermode="closest",
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.04,
        "xanchor": "center",
        "x": 0.5
    },
    margin={
        "t": 140,
        "b": 80,
        "l": 70,
        "r": 40
    }
)

dashboard.show()

In [25]:
RUTA_RESULTADOS = Path("resultados")
RUTA_RESULTADOS.mkdir(exist_ok=True)

RUTA_DASHBOARD = (
    RUTA_RESULTADOS
    / "dashboard_aqualimpia.html"
)

dashboard.write_html(
    RUTA_DASHBOARD,
    include_plotlyjs=True,
    full_html=True
)

print(
    "Dashboard guardado en:",
    RUTA_DASHBOARD
)

Dashboard guardado en: resultados\dashboard_aqualimpia.html


In [35]:
reporte_operaciones = (
    funciones.crear_reporte_operaciones(
        df_original
    )
)

reporte_gestion_ambiental = (
    funciones.crear_reporte_gestion_ambiental(
        df_original
    )
)

print(
    "Reporte de Operaciones:",
    reporte_operaciones.shape
)

print(
    "Reporte de Gestión Ambiental:",
    reporte_gestion_ambiental.shape
)

display(
    reporte_operaciones.head(3)
)

display(
    reporte_gestion_ambiental.head(3)
)

Reporte de Operaciones: (200, 9)
Reporte de Gestión Ambiental: (200, 5)


,fecha_registro,planta,caudal_entrada_m3_d,DBO_entrada_mg_L,DBO_salida_mg_L,eficiencia_remocion_DBO_pct,energia_aeracion_kWh,lodos_generados_kg_d,lodos_especificos_kg_m3
0,2025-07-01,Planta Norte,4545,432,79.0,81.712963,1450.1,399.3,0.087855
1,2025-07-02,Planta Norte,5860,235,32.0,86.382979,1528.4,494.0,0.084300
2,2025-07-03,Planta Centro,5124,396,64.9,83.611111,1035.3,459.8,0.089735


,fecha_registro,planta,DBO_salida_mg_L,cumplimiento_norma,cumplimiento_registrado
0,2025-07-01,Planta Norte,79.0,0,Incumplimiento registrado
1,2025-07-02,Planta Norte,32.0,0,Incumplimiento registrado
2,2025-07-03,Planta Centro,64.9,0,Incumplimiento registrado


In [36]:
rutas_reportes = (
    funciones.guardar_reportes_excel(
        df_original,
        RUTA_RESULTADOS
    )
)

for nombre, ruta in rutas_reportes.items():
    print(
        f"{nombre}: {ruta} | "
        f"existe: {ruta.exists()}"
    )

operaciones: resultados\reporte_operaciones.xlsx | existe: True
gestion_ambiental: resultados\reporte_gestion_ambiental.xlsx | existe: True


In [29]:
reporte_operaciones_verificado = pd.read_excel(
    rutas_reportes["operaciones"]
)

reporte_ambiental_verificado = pd.read_excel(
    rutas_reportes["gestion_ambiental"]
)

pd.testing.assert_frame_equal(
    reporte_operaciones_verificado,
    reporte_operaciones,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10
)

pd.testing.assert_frame_equal(
    reporte_ambiental_verificado,
    reporte_gestion_ambiental,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10
)

print(
    "Reportes verificados correctamente."
)

Reportes verificados correctamente.


In [37]:
RUTA_RESUMEN_JOBLIB = (
    RUTA_RESULTADOS
    / "resumen_resultados.joblib"
)

ruta_resumen_guardado = (
    funciones.guardar_resumen_joblib(
        df_original,
        RUTA_RESUMEN_JOBLIB
    )
)

resumen_esperado = (
    funciones.crear_resumen_resultados(
        df_original
    )
)

resumen_recuperado = (
    funciones.cargar_resumen_joblib(
        RUTA_RESUMEN_JOBLIB
    )
)

pd.testing.assert_frame_equal(
    resumen_recuperado["resumen_por_planta"],
    resumen_esperado["resumen_por_planta"]
)

for clave in [
    "cantidad_registros",
    "cantidad_plantas",
    "fecha_inicial",
    "fecha_final"
]:
    assert (
        resumen_recuperado[clave]
        == resumen_esperado[clave]
    )

print(
    "Archivo guardado:",
    ruta_resumen_guardado
)

print(
    "Existe:",
    ruta_resumen_guardado.exists()
)

print(
    "Claves recuperadas:",
    list(resumen_recuperado.keys())
)

display(
    resumen_recuperado[
        "resumen_por_planta"
    ].round(2)
)

Archivo guardado: resultados\resumen_resultados.joblib
Existe: True
Claves recuperadas: ['cantidad_registros', 'cantidad_plantas', 'fecha_inicial', 'fecha_final', 'resumen_por_planta']


,planta,registros,DBO_salida_promedio_mg_L,eficiencia_promedio_pct,eficiencia_mediana_pct,cumplimiento_registrado_pct
0,Planta Centro,75,35.90,87.51,87.76,22.67
1,Planta Norte,71,36.56,86.65,86.72,16.90
2,Planta Sur,54,36.06,87.10,86.73,29.63


### 5.1 Persistencia de resultados con Joblib

El archivo `resultados/resumen_resultados.joblib` conserva los principales resultados calculados, incluido el periodo analizado, la cantidad de registros, el número de plantas y el resumen comparativo por instalación. Su propósito es permitir la recuperación de estos objetos sin repetir el procesamiento completo.

Joblib no reemplaza los reportes Excel ni el dashboard, porque su contenido está destinado a reutilización programática y no a lectura directa. Por seguridad, el archivo solo debe cargarse cuando proviene de una fuente confiable, ya que los formatos de serialización de objetos pueden ejecutar contenido malicioso durante su lectura.

La prueba realizada guardó el resumen, lo recuperó desde el archivo y comparó sus valores con el objeto original. No se detectaron diferencias.